# Seminar 2. Custom PyTorch Operators

# Building Models in PyTorch Through Composition

PyTorch models are built using **composition**.  
Instead of defining one large monolithic network, we construct models by combining smaller, reusable modules.

Each module can contain other modules, which allows us to build hierarchical and well-structured architectures.

---

## Composition

Composition means:

- A model is built from smaller blocks.
- Each block can contain multiple layers.
- Blocks can be reused in larger architectures.
- Complex models are created by stacking simpler components.

This keeps code:

- Modular  
- Reusable  
- Readable  
- Easy to extend  


## Key Ideas

- Inherit from `nn.Module`
- Define layers inside `__init__`
- Define computation in `forward()`
- Create reusable blocks
- Build larger models by combining blocks

---

## Example: Model Built from Two Blocks

Below is a simple example where:

- We define a reusable blocks: `LinearReLUBlock` and `LinearTanhBlock`
- The final model is composed of two such blocks


In [1]:
import torch
import torch.nn as nn


class LinearReLUBlock(nn.Module):
    def __init__(self, in_features: int, out_features: int):
        super().__init__()

        self.linear = nn.Linear(in_features, out_features)
        self.activation = nn.ReLU()

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.linear(x)
        x = self.activation(x)
        return x


class LinearTanhBlock(nn.Module):
    def __init__(self, in_features: int, out_features: int):
        super().__init__()

        self.linear = nn.Linear(in_features, out_features)
        self.activation = nn.Tanh()

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.linear(x)
        x = self.activation(x)
        return x


class CombinedModel(nn.Module):
    def __init__(self):
        super().__init__()

        self.block1 = LinearReLUBlock(4, 8)
        self.block2 = LinearTanhBlock(8, 8)
        self.output = nn.Linear(8, 2)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.block1(x)
        x = self.block2(x)
        x = self.output(x)
        return x


model = CombinedModel()
print(model)

CombinedModel(
  (block1): LinearReLUBlock(
    (linear): Linear(in_features=4, out_features=8, bias=True)
    (activation): ReLU()
  )
  (block2): LinearTanhBlock(
    (linear): Linear(in_features=8, out_features=8, bias=True)
    (activation): Tanh()
  )
  (output): Linear(in_features=8, out_features=2, bias=True)
)


# What `nn.Module` Enables

When we inherit from `nn.Module`, we automatically gain powerful functionality that works **recursively across all submodules**.

## What `nn.Module` Gives Us



### Parameter Registration

All layers assigned as attributes (e.g. `self.linear = nn.Linear(...)`) are:

- Automatically registered
- Collected in `model.parameters()`
- Included in `model.state_dict()`

This works **recursively** for all sub-blocks.

In [2]:
print("Registered parameters:")
for name, param in model.named_parameters():
    print(name, param.shape)


Registered parameters:
block1.linear.weight torch.Size([8, 4])
block1.linear.bias torch.Size([8])
block2.linear.weight torch.Size([8, 8])
block2.linear.bias torch.Size([8])
output.weight torch.Size([2, 8])
output.bias torch.Size([2])


### Automatic Gradient Tracking

During the forward pass:

- PyTorch dynamically builds a computation graph
- Calling `loss.backward()` computes gradients
- Gradients are stored in each parameter’s `.grad`

No manual graph management is required.

In [3]:
x = torch.randn(5, 4)
target = torch.randn(5, 2)

criterion = nn.MSELoss()
output = model(x)
loss = criterion(output, target)

loss.backward()

print("\nGradient computed for output layer:",
      model.output.weight.grad is not None)
model.output.weight.grad


Gradient computed for output layer: True


tensor([[-0.0007,  0.0451, -0.0230, -0.0433,  0.0253,  0.0198,  0.0380, -0.1098],
        [-0.1221,  0.0386, -0.0797, -0.0710,  0.0595,  0.0542,  0.0863, -0.0861]])

### Device and Type Transfer (`.to()`)

Calling:

    model.to(device)

or

    model.to(dtype)

moves all:

- Parameters
- Buffers
- Submodules

to CPU/GPU/dtype automatically.

In [4]:
import torch

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

dtype = torch.float32

model = model.to(device=device, dtype=dtype)

x: torch.Tensor = torch.randn(5, 4, device=device, dtype=dtype)

print("Model device:", next(model.parameters()).device)
print("Model dtype:", next(model.parameters()).dtype)

Model device: cpu
Model dtype: torch.float32


### Saving & Loading (`state_dict()`)

- `model.state_dict()` returns all parameters recursively
- `model.load_state_dict(...)` restores them

This works across the full module tree.

In [5]:
state_dict = model.state_dict()
torch.save(state_dict, "combined_model.pt")

# `train()` vs `eval()` Mode in PyTorch

PyTorch modules have two main modes: **training mode** and **evaluation mode**.  
Switching between them affects layers that behave differently during training and inference.

---

## `model.train()`

- Sets the model to **training mode**.
- Used when training the model with gradient updates.
- Affects certain layers, such as:

| Layer Type        | Behavior in `train()` Mode                  |
|------------------|--------------------------------------------|
| `Dropout`         | Randomly zeroes some activations           |
| `BatchNorm`       | Updates running statistics (mean/variance) |

- Gradients are computed as usual.

---

## `model.eval()`

- Sets the model to **evaluation (inference) mode**.
- Used when evaluating or deploying the model.
- Affects certain layers:

| Layer Type        | Behavior in `eval()` Mode                   |
|------------------|--------------------------------------------|
| `Dropout`         | Passes all activations through unchanged  |
| `BatchNorm`       | Uses stored running mean/variance         |

- No layers update internal statistics.
- Gradients are usually not required (often used with `torch.no_grad()`).

---

## Key Points

- Always use `model.train()` during training.
- Always use `model.eval()` during evaluation or testing.
- Forgetting to switch can lead to inconsistent results, especially with `Dropout` or `BatchNorm`.



In [6]:
import torch
import torch.nn as nn

# Simple model with Dropout and BatchNorm
class SimpleModel(nn.Module):
    def __init__(self) -> None:
        super().__init__()
        self.fc1 = nn.Linear(4, 8)
        self.bn = nn.BatchNorm1d(8)
        self.dropout = nn.Dropout(p=0.5)
        self.fc2 = nn.Linear(8, 2)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.fc1(x)
        x = self.bn(x)
        x = torch.relu(x)
        x = self.dropout(x)
        x = self.fc2(x)
        return x


model = SimpleModel()
x = torch.randn(5, 4)

# Training mode
model.train()
output_train = model(x)
print("Training mode output:\n", output_train)

# Evaluation mode
model.eval()
with torch.no_grad():
    output_eval = model(x)
print("Evaluation mode output:\n", output_eval)


Training mode output:
 tensor([[ 0.0233,  0.1153],
        [-0.4952, -1.4590],
        [-0.2690, -0.2023],
        [-0.4976,  0.4965],
        [ 0.0891, -1.5299]], grad_fn=<AddmmBackward0>)
Evaluation mode output:
 tensor([[ 0.0118,  0.0811],
        [-0.2875, -0.3415],
        [-0.1265,  0.2187],
        [-0.1961,  0.1932],
        [-0.0964, -0.4636]])


# `torch.no_grad()` and `torch.inference_mode()` in PyTorch

When performing inference (evaluating a model without updating parameters), PyTorch provides context managers to **disable gradient tracking**. This saves memory and speeds up computation.

---

## `torch.no_grad()`

- Disables gradient tracking.
- Useful during evaluation or inference.
- Gradients are **not computed**, but autograd still tracks operations for some internal purposes.
- Can be used as a **context manager** or a **function decorator**.

---

## `torch.inference_mode()`

- Introduced in PyTorch 1.9.
- Similar to `no_grad()`, but **more efficient**.
- Completely disables autograd and reduces memory usage.
- Recommended for pure inference pipelines.



In [8]:
import torch
import torch.nn as nn

class SimpleModel(nn.Module):
    def __init__(self) -> None:
        super().__init__()
        self.fc = nn.Linear(4, 2)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.fc(x)

    # -----------------------------
    # Using torch.no_grad() as method decorator
    # -----------------------------
    @torch.no_grad()
    def forward_no_grad(self, x: torch.Tensor) -> torch.Tensor:
        return self.fc(x)

    # -----------------------------
    # Using torch.inference_mode() as method decorator
    # -----------------------------
    @torch.inference_mode()
    def forward_inference(self, x: torch.Tensor) -> torch.Tensor:
        return self.fc(x)


model = SimpleModel()
x = torch.randn(5, 4)

# -----------------------------
# Call decorated methods
# -----------------------------
output_no_grad_method = model.forward_no_grad(x)
output_infer_method = model.forward_inference(x)

print("Output no_grad method:\n", output_no_grad_method)
print("Output inference_mode method:\n", output_infer_method)

# -----------------------------
# Using context managers
# -----------------------------

with torch.no_grad():
    output_no_grad_cm = model(x)

with torch.inference_mode():
    output_infer_cm = model(x)

print("Output no_grad context manager:\n", output_no_grad_cm)
print("Output inference_mode context manager:\n", output_infer_cm)


Output no_grad method:
 tensor([[ 0.2347,  0.7337],
        [ 0.7393, -0.6275],
        [-0.6316,  0.2326],
        [-0.0753,  0.6684],
        [ 0.4330,  0.1811]])
Output inference_mode method:
 tensor([[ 0.2347,  0.7337],
        [ 0.7393, -0.6275],
        [-0.6316,  0.2326],
        [-0.0753,  0.6684],
        [ 0.4330,  0.1811]])
Output no_grad context manager:
 tensor([[ 0.2347,  0.7337],
        [ 0.7393, -0.6275],
        [-0.6316,  0.2326],
        [-0.0753,  0.6684],
        [ 0.4330,  0.1811]])
Output inference_mode context manager:
 tensor([[ 0.2347,  0.7337],
        [ 0.7393, -0.6275],
        [-0.6316,  0.2326],
        [-0.0753,  0.6684],
        [ 0.4330,  0.1811]])


In [14]:
print(output_no_grad_cm.is_inference())
print(output_infer_cm.is_inference())

False
True


# Disabling Gradients with `requires_grad_(False)`

PyTorch provides a convenient method `requires_grad_()` that can **enable or disable gradients in-place** for all parameters of a model or a tensor.

Using:

```python
param.requires_grad_(False)
```

- Sets `requires_grad=False` **in-place** for that parameter.
- This is useful for freezing models during inference or transfer learning.
- Can be applied to an entire model recursively by iterating over its parameters.


In [15]:
model = SimpleModel()

# Disable gradient computation for all parameters using requires_grad_()
for param in model.parameters():
    param.requires_grad_(False)

# Verify
for name, param in model.named_parameters():
    print(f"{name}: requires_grad={param.requires_grad}")

# Forward pass still works
x = torch.randn(5, 4)
output = model(x)
print("Output shape:", output.shape)

fc.weight: requires_grad=False
fc.bias: requires_grad=False
Output shape: torch.Size([5, 2])


# Redefining `train()` and `eval()` in `nn.Module`

PyTorch’s `nn.Module` provides built-in `train(mode: bool = True)` and `eval()` methods to switch between **training** and **evaluation** modes.  

Sometimes, when creating **custom modules or blocks**, you might want to **override these methods** to perform extra actions whenever the mode changes.

---

## Why Override?

- Apply mode-specific logic to sub-blocks or attributes that are not standard layers
- Log or track mode switches
- Automatically modify internal flags or buffers along with training/eval mode

---

## How It Works

- `train(mode: bool = True)` sets `self.training = mode` for the module
- `eval()` is equivalent to `train(False)`
- Default implementation recursively calls `train(mode)` on all submodules
- Overriding allows custom behavior while keeping recursive updates intact


In [16]:
import torch
import torch.nn as nn
from torch import Tensor
from typing import Self

class CustomBlock(nn.Module):
    def __init__(self) -> None:
        super().__init__()
        self.linear = nn.Linear(4, 4)
        self.dropout = nn.Dropout(p=0.5)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.linear(x)
        x = torch.relu(x)
        x = self.dropout(x)
        return x

    # -----------------------------
    # Override train() method
    # -----------------------------
    def train(self, mode: bool = True) -> Self:
        print(f"CustomBlock set to {'train' if mode else 'eval'} mode")
        super().train(mode)  # Call original method to update submodules
        # Add any custom logic here
        return self

    # -----------------------------
    # Override eval() method
    # -----------------------------
    def eval(self) -> Self:
        print("CustomBlock set to eval mode")
        return super().eval()


# Example usage
model = CustomBlock()
x = torch.randn(2, 4)

# Switch to training mode
model.train()
output_train = model(x)

# Switch to evaluation mode
model.eval()
with torch.no_grad():
    output_eval = model(x)

print("Output training mode:", output_train)
print("Output eval mode:", output_eval)


CustomBlock set to train mode
CustomBlock set to eval mode
CustomBlock set to eval mode
Output training mode: tensor([[0.0000, 0.0000, 1.3484, 0.0000],
        [0.0000, 0.5489, 0.0000, 0.0000]], grad_fn=<MulBackward0>)
Output eval mode: tensor([[0.0000, 0.0000, 0.6742, 0.0000],
        [0.0000, 0.2744, 0.3912, 0.0000]])


# Common Module Aggregators in PyTorch

When building neural networks, it is often useful to group multiple layers or submodules together.  
PyTorch provides several **module aggregators** that help organize layers and blocks. The most common ones are:



## `nn.Sequential`

- Holds modules in a sequential order.
- Executes them **in the order they are added** during the forward pass.
- Ideal for simple **stacked layers** with a single input and output.

**Key points:**

- Forward pass is automatically defined.
- Cannot handle multiple inputs or branching.

In [17]:
seq_model = nn.Sequential(
    nn.Linear(4, 8),
    nn.ReLU(),
    nn.Linear(8, 2)
)

x = torch.randn(5, 4)
output_seq = seq_model(x)
print("nn.Sequential output shape:", output_seq.shape)


nn.Sequential output shape: torch.Size([5, 2])


## `nn.ModuleList`

- Holds a **list of modules**.
- Does **not define a forward pass automatically**.
- Useful when you need to **loop over modules**, or have conditional computation.

**Key points:**

- Modules are registered properly, so parameters are tracked.
- You must define your own `forward()`.

In [18]:
class ModuleListModel(nn.Module):
    def __init__(self) -> None:
        super().__init__()
        self.layers = nn.ModuleList([
            nn.Linear(4, 8),
            nn.ReLU(),
            nn.Linear(8, 2)
        ])

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        for layer in self.layers:
            x = layer(x)
        return x

ml_model = ModuleListModel()
output_ml = ml_model(x)
print("nn.ModuleList output shape:", output_ml.shape)

nn.ModuleList output shape: torch.Size([5, 2])


## `nn.ModuleDict`

- Holds modules in a **dictionary** with string keys.
- Useful for architectures with **named branches**, **dynamic selection**, or **multi-head outputs**.
- Like `ModuleList`, it does **not define a forward pass**.

In [19]:
class ModuleDictModel(nn.Module):
    def __init__(self) -> None:
        super().__init__()
        self.branches = nn.ModuleDict({
            "branch1": nn.Linear(4, 8),
            "branch2": nn.Linear(4, 8)
        })
        self.output: nn.Linear = nn.Linear(8, 2)

    def forward(self, x: torch.Tensor, branch_name: str = "branch1") -> torch.Tensor:
        x = self.branches[branch_name](x)
        return self.output(x)

md_model = ModuleDictModel()
output_md = md_model(x, branch_name="branch2")
print("nn.ModuleDict output shape:", output_md.shape)

nn.ModuleDict output shape: torch.Size([5, 2])


## Type hints and Shape Information

For production code, a practical pattern is to use `Tensor` for type hints and document the expected shape next to the signature.

`Tensor` tells us what kind of object is expected, while `[B, C, H, W]` documents the tensor dimensions explicitly.


In [20]:
from torch import Tensor

def normalize_images(x: Tensor) -> Tensor:
    # x: [B, C, H, W]
    return x / 255.0



## Работа на семинаре

### LSTM

![lstm](https://github.com/sn0rkmaiden/MTAI2026-Deep-learning-frameworks-seminars-autumn/blob/main/seminar_2/assets/LSTM.png?raw=1)

In [25]:
class LSTMCell(nn.Module):
    """One LSTM step: (x_t, (h_{t-1}, c_{t-1})) -> (h_t, c_t)."""

    def __init__(self, input_size: int, hidden_size: int) -> None:
        super().__init__()
        self.hidden_size = hidden_size
        self.W_f = nn.Linear(input_size + hidden_size, hidden_size)
        self.W_c = nn.Linear(input_size + hidden_size, hidden_size)
        self.W_i = nn.Linear(input_size + hidden_size, hidden_size)
        self.W_o = nn.Linear(input_size + hidden_size, hidden_size)

    def forward(
        self, x_t: Tensor, state: tuple[Tensor, Tensor]
    ) -> tuple[Tensor, Tensor]:
        h_prev, c_prev = state
        combined = torch.cat([h_prev, x_t], dim=-1)   # [B, hidden+input]

        f_t = torch.sigmoid(self.W_f(combined))        # forget gate
        c_tilde = torch.tanh(self.W_c(combined))        # candidate cell content
        i_t = torch.sigmoid(self.W_i(combined))         # input gate
        o_t = torch.sigmoid(self.W_o(combined))         # output gate

        c_t = f_t * c_prev + i_t * c_tilde
        h_t = o_t * torch.tanh(c_t)

        return h_t, c_t


class LSTM(nn.Module):
    """Runs LSTMCell over a whole sequence."""

    def __init__(self, input_size: int, hidden_size: int) -> None:
        super().__init__()
        self.hidden_size = hidden_size
        self.cell = LSTMCell(input_size, hidden_size)

    def forward(
        self, x: Tensor, state: tuple[Tensor, Tensor] | None = None
    ) -> tuple[Tensor, tuple[Tensor, Tensor]]:
        # x: [B, T, input_size]
        batch_size, seq_len, _ = x.shape
        if state is None:
            h_t = x.new_zeros(batch_size, self.hidden_size)
            c_t = x.new_zeros(batch_size, self.hidden_size)
        else:
            h_t, c_t = state

        outputs = []
        for t in range(seq_len):
            h_t, c_t = self.cell(x[:, t, :], (h_t, c_t))
            outputs.append(h_t)

        return torch.stack(outputs, dim=1), (h_t, c_t)   # [B, T, hidden], final state


if __name__ == "__main__":
    batch, seq_len, input_size, hidden_size = 3, 5, 4, 6
    x = torch.randn(batch, seq_len, input_size)

    my_lstm = LSTM(input_size, hidden_size)
    out, (h_n, c_n) = my_lstm(x)
    print("Output shape:", out.shape)
    print("h_n shape:", h_n.shape)
    print("c_n shape:", c_n.shape)

Output shape: torch.Size([3, 5, 6])
h_n shape: torch.Size([3, 6])
c_n shape: torch.Size([3, 6])


### Inception

![inception](https://github.com/sn0rkmaiden/MTAI2026-Deep-learning-frameworks-seminars-autumn/blob/main/seminar_2/assets/inception.png?raw=1)

In [26]:
class ConvBlock(nn.Module):
    """Conv2d + BatchNorm + ReLU, a common building block."""

    def __init__(self, in_channels: int, out_channels: int, **kwargs) -> None:
        super().__init__()
        self.conv = nn.Conv2d(in_channels, out_channels, bias=False, **kwargs)
        self.bn = nn.BatchNorm2d(out_channels)
        self.relu = nn.ReLU(inplace=True)

    def forward(self, x: Tensor) -> Tensor:
        return self.relu(self.bn(self.conv(x)))


class InceptionModule(nn.Module):
    """
    Inception module with dimension reduction (GoogLeNet, Fig. 3b).

    Four parallel branches, concatenated along the channel dimension:
      1) 1x1 conv
      2) 1x1 conv -> 3x3 conv
      3) 1x1 conv -> 5x5 conv
      4) 3x3 max pool -> 1x1 conv
    """

    def __init__(
        self,
        in_channels: int,
        out_1x1: int,
        reduce_3x3: int,
        out_3x3: int,
        reduce_5x5: int,
        out_5x5: int,
        pool_proj: int,
    ) -> None:
        super().__init__()

        self.branch1 = ConvBlock(in_channels, out_1x1, kernel_size=1)
        self.branch2 = nn.Sequential(
            ConvBlock(in_channels, reduce_3x3, kernel_size=1),
            ConvBlock(reduce_3x3, out_3x3, kernel_size=3, padding=1),
        )
        self.branch3 = nn.Sequential(
            ConvBlock(in_channels, reduce_5x5, kernel_size=1),
            ConvBlock(reduce_5x5, out_5x5, kernel_size=5, padding=2),
        )
        self.branch4 = nn.Sequential(
            nn.MaxPool2d(kernel_size=3, stride=1, padding=1),
            ConvBlock(in_channels, pool_proj, kernel_size=1),
        )

    def forward(self, x: Tensor) -> Tensor:
        b1 = self.branch1(x)
        b2 = self.branch2(x)
        b3 = self.branch3(x)
        b4 = self.branch4(x)
        return torch.cat([b1, b2, b3, b4], dim=1)


if __name__ == "__main__":
    x = torch.randn(2, 192, 28, 28)  # [B, C, H, W]
    module = InceptionModule(
        in_channels=192,
        out_1x1=64,
        reduce_3x3=96, out_3x3=128,
        reduce_5x5=16, out_5x5=32,
        pool_proj=32,
    )
    out = module(x)
    print("Output shape:", out.shape)  # [2, 64+128+32+32, 28, 28] = [2, 256, 28, 28]

Output shape: torch.Size([2, 256, 28, 28])


### SE

![se](https://github.com/sn0rkmaiden/MTAI2026-Deep-learning-frameworks-seminars-autumn/blob/main/seminar_2/assets/SqueezeAndExcite.png?raw=1)

In [27]:
class SEBlock(nn.Module):
    """
    Squeeze-and-Excitation block (Hu et al., 2018).

    U -> squeeze (global avg pool) -> excitation (bottleneck MLP + sigmoid)
      -> scale (channel-wise reweighting) -> X_tilde
    """

    def __init__(self, channels: int, reduction: int = 16) -> None:
        super().__init__()
        reduced = max(channels // reduction, 1)

        #[B, C, H, W] -> [B, C, 1, 1]
        self.squeeze = nn.AdaptiveAvgPool2d(1)
        self.excitation = nn.Sequential(
            nn.Linear(channels, reduced, bias=False),
            nn.ReLU(inplace=True),
            nn.Linear(reduced, channels, bias=False),
            nn.Sigmoid(),
        )

    def forward(self, u: Tensor) -> Tensor:
        b, c, _, _ = u.shape

        # [B, C, H, W] -> [B, C]
        z = self.squeeze(u).view(b, c)

        # [B, C] -> [B, C] channel weights
        s = self.excitation(z)

        s = s.view(b, c, 1, 1)
        return u * s


if __name__ == "__main__":
    x = torch.randn(4, 64, 32, 32)  # [B, C, H, W]
    se = SEBlock(channels=64, reduction=16)
    out = se(x)
    print("Output shape:", out.shape)

Output shape: torch.Size([4, 64, 32, 32])


### Selective Kernel

![selective](https://github.com/sn0rkmaiden/MTAI2026-Deep-learning-frameworks-seminars-autumn/blob/main/seminar_2/assets/SelectiveKernel.png?raw=1)


In [29]:
class SKConv(nn.Module):
    """
    Selective Kernel Convolution (Li et al., 2019).

    Split: M parallel conv branches with different receptive fields.
    Fuse:  elementwise sum -> global avg pool -> FC to a compact vector z.
    Select: per-branch FC -> softmax across branches -> weighted sum.
    """

    def __init__(
        self,
        channels: int,
        M: int = 2,
        reduction: int = 16,
        L: int = 32,
        stride: int = 1,
    ) -> None:
        super().__init__()
        d = max(channels // reduction, L)
        self.M = M
        self.channels = channels

        self.branches = nn.ModuleList([
            nn.Sequential(
                nn.Conv2d(
                    channels, channels, kernel_size=3, stride=stride,
                    padding=1 + i, dilation=1 + i, groups=32, bias=False,
                ),
                nn.BatchNorm2d(channels),
                nn.ReLU(inplace=True),
            )
            for i in range(M)
        ])

        self.fc = nn.Sequential(
            nn.Linear(channels, d),
            nn.BatchNorm1d(d),
            nn.ReLU(inplace=True),
        )

        self.fcs = nn.ModuleList([nn.Linear(d, channels) for _ in range(M)])
        self.softmax = nn.Softmax(dim=1)

    def forward(self, x: Tensor) -> Tensor:
        b, c, h, w = x.shape

        feats = [branch(x) for branch in self.branches]     # M x [B, C, H, W]
        feats = torch.stack(feats, dim=1)                    # [B, M, C, H, W]

        u = feats.sum(dim=1)                                  # [B, C, H, W]
        s = u.mean(dim=[2, 3])                                # F_gp: [B, C]
        z = self.fc(s)                                        # F_fc: [B, d]

        attn = torch.stack([fc(z) for fc in self.fcs], dim=1)  # [B, M, C]
        attn = self.softmax(attn)                               # softmax over M, per channel
        attn = attn.view(b, self.M, c, 1, 1)                    # broadcast over H, W

        v = (feats * attn).sum(dim=1)                           # [B, C, H, W]
        return v

if __name__ == "__main__":
    x = torch.randn(4, 64, 32, 32)  # [B, C, H, W]
    sk = SKConv(channels=64, M=2, reduction=16, L=32)
    out = sk(x)
    print("Output shape:", out.shape)  # [4, 64, 32, 32]

Output shape: torch.Size([4, 64, 32, 32])


### PatchMerger

![patchmerger](https://github.com/sn0rkmaiden/MTAI2026-Deep-learning-frameworks-seminars-autumn/blob/main/seminar_2/assets/PatchMerger.png?raw=1)


In [30]:
class PatchMerger(nn.Module):
    """
    PatchMerger (Renggli et al., 2022): reduces N input tokens to M output
    tokens, each a learned attention-weighted combination of the input tokens.

    Input:  [B, N, D]
    Output: [B, M, D]
    """

    def __init__(self, dim: int, num_tokens_out: int) -> None:
        super().__init__()
        self.norm = nn.LayerNorm(dim)
        self.weight = nn.Parameter(torch.randn(num_tokens_out, dim) * (dim ** -0.5))

    def forward(self, x: Tensor) -> Tensor:
        x_norm = self.norm(x)
        scores = x_norm @ self.weight.t()
        attn = scores.transpose(1, 2).softmax(dim=-1)
        out = attn @ x_norm
        return out

if __name__ == "__main__":
    batch, n_tokens, dim, m_tokens = 2, 196, 768, 8
    x = torch.randn(batch, n_tokens, dim)

    merger = PatchMerger(dim=dim, num_tokens_out=m_tokens)
    out = merger(x)
    print("Output shape:", out.shape)  # [2, 8, 768]

Output shape: torch.Size([2, 8, 768])


## Домашнее задание

## Задание 1, реализуйте требуемые блоки

## ResNet Block (0.1 балл)

![Resnet](https://github.com/sn0rkmaiden/MTAI2026-Deep-learning-frameworks-seminars-autumn/blob/main/seminar_2/assets/ResBlock.png?raw=1)

https://arxiv.org/pdf/1512.03385

In [32]:
class ResidualBlock(nn.Module):
    def __init__(self, in_channels, out_channels, stride: int=1):
        super().__init__()
        self.conv1 = nn.Conv2d(
            in_channels, out_channels, kernel_size=3,
            stride=stride, padding=1, bias=False,
        )
        self.bn1 = nn.BatchNorm2d(out_channels)
        self.relu = nn.ReLU(inplace=True)

        # if shapes already match
        if stride != 1 or in_channels != out_channels:
            self.shortcut = nn.Sequential(
                nn.Conv2d(
                    in_channels, out_channels, kernel_size=1,
                    stride=stride, bias=False,
                ),
                nn.BatchNorm2d(out_channels),
            )
        else:
            self.shortcut = nn.Identity()

    def forward(self, x):
        identity = self.shortcut(x)

        out = self.conv1(x)
        out = self.bn1(out)
        out = self.relu(out)

        out = self.conv2(out)
        out = self.bn2(out)

        out = out + identity   # skip connection
        out = self.relu(out)
        return out

## Depthwise Separable Convolution (0.1 балл)
![DepthWiseConv](https://github.com/sn0rkmaiden/MTAI2026-Deep-learning-frameworks-seminars-autumn/blob/main/seminar_2/assets/DepthWiseConv.png?raw=1)

https://arxiv.org/pdf/1610.02357

In [33]:
class SeparableConv2d(nn.Module):
    def __init__(self, in_channels, out_channels, kernel_size=3, stride=1, padding=1, bias=False):
        super().__init__()
        self.depthwise = nn.Conv2d(
            in_channels, in_channels, kernel_size=kernel_size,
            stride=stride, padding=padding, groups=in_channels, bias=bias,
        )

        self.pointwise = nn.Conv2d(
            in_channels, out_channels, kernel_size=1, bias=bias,
        )

    def forward(self, x):
        x = self.depthwise(x)
        x = self.pointwise(x)
        return x

## Vanilla Attention (0.1 балл)

Let:

$$
\text{query} \in \mathbb{R}^{B \times d} \\
\text{key} \in \mathbb{R}^{B \times L \times d}
$$

---

### Alignment Scores

$$
\text{score} = \text{key} \cdot (W_\text{align} \, \text{query})^T \\
\text{score} \in \mathbb{R}^{B \times L}
$$

---

### Attention Weights

$$
\text{att} = \text{softmax}(\text{score}, \text{dim}=1) \\
\text{att} \in \mathbb{R}^{B \times L}
$$

---

### Context Vector

$$
\text{context} = \sum_{i=1}^{L} \text{att}_i \cdot \text{key}_i \\
\text{context} \in \mathbb{R}^{B \times d}
$$

---

### Output

$$
\text{out} = \tanh(W_\text{value} \, \text{context} + W_\text{query} \, \text{query}) \\
\text{out} \in \mathbb{R}^{B \times d}
$$



https://arxiv.org/abs/1409.0473


https://arxiv.org/abs/1508.04025

In [34]:
from typing import Optional
import torch
from torch import nn
import numpy as np

class VanillaAttention(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.W_align = nn.Linear(dim, dim, bias=False)
        self.W_value = nn.Linear(dim, dim, bias=False)
        self.W_query = nn.Linear(dim, dim, bias=False)
        self.softmax = nn.Softmax(dim=1)


    def forward(self, query, key):
        aligned_query = self.W_align(query)
        score = torch.bmm(key, aligned_query.unsqueeze(-1)).squeeze(-1)
        att = self.softmax(score)
        context = torch.bmm(att.unsqueeze(1), key).squeeze(1)
        out = torch.tanh(self.W_value(context) + self.W_query(query))
        return out

## Dot Product Attention (0.1 балл)

$$
Q \in \mathbb{R}^{B \times L_q \times d_k} \\
K \in \mathbb{R}^{B \times L_k \times d_k} \\
V \in \mathbb{R}^{B \times L_k \times d_k}
$$

$$
S = \frac{Q K^T}{\sqrt{d_k}}
$$

$$
\text{Attention}(Q, K, V) = \text{softmax}(S, \text{dim}=-1) \, V
$$



https://arxiv.org/abs/1706.03762


In [35]:
class ScaledDotProductAttention(nn.Module):
    def __init__(self, dropout):
        super().__init__()
        self.sotfmax = nn.Softmax(dim=-1)
        self.dropout = nn.Dropout(dropout=dropout)

    def forward(self, Q, K, V, mask):
        d_k = Q.size(-1)
        S = torch.bmm(Q, K.transpose(-2, -1)) / np.sqrt(d_k)
        if mask is not None:
          if mask.dtype == torch.bool:
            S = S.masked_fill(~mask, float("-inf"))
          else:
            S = S + mask

        attn = self.softmax(S)
        attn = self.dropout(attn)
        out = torch.bmm(attn, V)
        return out

## Multihead Attention (0.1 балл)

![MultiheadAttention](https://github.com/sn0rkmaiden/MTAI2026-Deep-learning-frameworks-seminars-autumn/blob/main/seminar_2/assets/MultiheadAttention.webp?raw=1)

https://arxiv.org/abs/1706.03762


In [37]:
class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, num_heads, dropout=0.0):
        super().__init__()
        assert d_model % num_heads == 0

        self.num_heads = num_heads
        self.d_k = d_model // num_heads

        self.W_v = nn.Linear(d_model, d_model)
        self.W_k = nn.Linear(d_model, d_model)
        self.W_q = nn.Linear(d_model, d_model)
        self.W_o = nn.Linear(d_model, d_model)

        self.attention = ScaledDotProductAttention(dropout=dropout)

    def _split_heads(self, x):
        B, L, _ = x.shape
        x = x.view(B, L, self.num_heads, self.d_k)
        return x.transpose(1, 2)

    def forward(self, value, key, query, mask):
        V = self._split_heads(self.W_v(value))
        K = self._split_heads(self.W_k(key))
        Q = self._split_heads(self.W_q(query))

        if mask is not None and mask.dim() == 3:
          mask = mask.unsqueeze(1)

        B, H, Lq, d_k = Q.shape
        Q_flat = Q.reshape(B * H, Lq, d_k)
        K_flat = K.reshape(B * H, K.size(2), d_k)
        V_flat = V.reshape(B * H, V.size(2), d_k)

        mask_flat = None
        if mask is not None:
            mask_flat = mask.expand(B, H, Lq, K.size(2)).reshape(B * H, Lq, K.size(2))

        heads = self.attention(Q_flat, K_flat, V_flat, mask_flat)
        heads = heads.view(B, H, Lq, d_k)

        concat = self._merge_heads(heads)
        return self.W_o(concat)

## Transformer Encoder Layer (0.1 балл)


![Transformer Encoder Layer](https://github.com/sn0rkmaiden/MTAI2026-Deep-learning-frameworks-seminars-autumn/blob/main/seminar_2/assets/TransformerEncoder.png?raw=1)


https://arxiv.org/abs/1706.03762

In [39]:
class MLP(nn.Module):
    def __init__(self, d_model, d_ff, dropout):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(d_model, d_ff),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(d_ff, d_model),
            nn.Dropout(dropout),
        )

    def forward(self, x):
        return self.net(x)

class TransformerEncoderLayer(nn.Module):
    def __init__(self, d_model, num_heads, d_ff, dropout=0.0):
        super().__init__()
        self.norm1 = nn.LayerNorm(d_model)
        self.self_attn = MultiHeadAttention(d_model, num_heads, dropout=dropout)
        self.norm2 = nn.LayerNorm(d_model)
        self.mlp = MLP(d_model, d_ff, dropout=dropout)

    def forward(self, x, mask):
        normed = self.norm1(x)
        attn_out = self.self_attn(value=normed, key=normed, query=normed, mask=mask)
        x = x + attn_out
        normed = self.norm2(x)
        mlp_out = self.mlp(normed)
        x = x + mlp_out
        return x

## MLP Mixer (0.1 балл)


![MLPMixer](https://github.com/sn0rkmaiden/MTAI2026-Deep-learning-frameworks-seminars-autumn/blob/main/seminar_2/assets/MLPMixer.png?raw=1)


https://arxiv.org/abs/2105.01601

In [41]:
class MixerMLP(nn.Module):
    def __init__(self, dim, hid_dim, dropout=0.0):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(dim, hid_dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hid_dim, dim),
            nn.Dropout(dropout),
        )

    def forward(self, x):
        return self.net(x)


class MLPMixerBlock(nn.Module):
    def __init__(self, num_patches, channels, token_hidden_dim, channel_hidden_dim, dropout=0.0):
        super().__init__()

        self.norm1 = nn.LayerNorm(channels)
        self.token_mlp = MixerMLP(num_patches, token_hidden_dim, dropout=dropout)

        self.norm2 = nn.LayerNorm(channels)
        self.channel_mlp = MixerMLP(channels, channel_hidden_dim, dropout=dropout)

    def forward(self, x):
        y = self.norm1(x)
        y = y.transpose(1, 2)
        y = self.token_mlp(y)
        y = y.transpose(1, 2)
        x = x + y

        z = self.norm2(x)
        z = self.channel_mlp(z)
        x = x + z
        return x

## ConvMixer (0.1 балл)

![ConvMixer](https://github.com/sn0rkmaiden/MTAI2026-Deep-learning-frameworks-seminars-autumn/blob/main/seminar_2/assets/ConvMixer.png?raw=1)


https://arxiv.org/abs/2201.09792

In [43]:
class ConvMixerBlock(nn.Module):
    def __init__(self, dim, kernel_size=9):
        super().__init__()
        self.depthwise = nn.Conv2d(
            dim, dim, kernel_size=kernel_size,
            padding=kernel_size // 2, groups=dim,
        )
        self.relu1 = nn.ReLU()
        self.bn1 = nn.BatchNorm2d(dim)
        self.pointwise = nn.Conv2d(dim, dim, kernel_size=1)
        self.relu2 = nn.ReLU()
        self.bn2 = nn.BatchNorm2d(dim)

    def forward(self, x):
        residual = x
        out = self.depthwise(x)
        out = self.relu1(out)
        out = self.bn1(out)
        out = out + residual

        out = self.pointwise(out)
        out = self.relu2(out)
        out = self.bn2(out)
        return out


class ConvMixer(nn.Module):
    def __init__(self, dim, depth, kernel_size, patch_size, in_channels, num_classes):
        super().__init__()
        self.stem = nn.Sequential(
            nn.Conv2d(in_channels, dim, kernel_size=patch_size, stride=patch_size),
            nn.ReLU(),
            nn.BatchNorm2d(dim),
        )

        self.blocks = nn.Sequential(
            *[ConvMixerBlock(dim, kernel_size=kernel_size) for _ in range(depth)]
        )

        self.head = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),
            nn.Flatten(),
            nn.Linear(dim, num_classes),
        )


    def forward(self, x):
      x = self.stem(x)
      x = self.blocks(x)
      x = self.head(x)
      return x


## Вопрос (0.2 балла)

Объясните, почему MLPMixer, ConvMixer может работать почти так же эффективно, как обычный Multihead Attention.

Напишите формулу, связывающую Multihead Attention, ConvMixer и MLPMixer

Опишите преимущества и недостатки между ConvMixer, MLPMixer и Multihead Attention

---

Ответ:

1) Все архитектуры решают задачу перемешивания информации между токенами. Отличаются они только тем, как задается матрица весов.

* Multihead Attention: $Y = \text{softmax}\left(\frac{QK^\top}{\sqrt{d_k}}\right)V = A(X)V$

* MLP-Mixer: $Y = WV, \qquad W \in \mathbb{R}^{L \times L}, \quad W \neq W(X)$

* ConvMixer: $W_{ij} = w_{i-j} \cdot \mathbb{1}\left[\,|i-j| \le \tfrac{k}{2}\,\right]$

2) Все три — частные случаи одной операции "смешивания" токенов:

$Y = W(X)\,X, \qquad W(X) \in \mathbb{R}^{L \times L}$

$W_{ij} =
\begin{cases}
\text{softmax}\!\left(\dfrac{q_i^\top k_j}{\sqrt{d_k}}\right) & \text{Attention} \\[8pt]
w_{ij} & \text{MLP-Mixer} \\[6pt]
w_{i-j}\cdot\mathbb{1}\!\left[|i-j|\le \tfrac{k}{2}\right] & \text{ConvMixer}
\end{cases}$

3)

Multi-Head Attention:

* Матрица смешивания зависит от входа — гибко подстраивается под контент.
* Схватывает глобальный контекст и дальние зависимости с первого слоя.

* Сложность $O(L^2 d)$ по времени и памяти.

* Требуется много данных.

MLP-Mixer:

* Проще реализовано внимание, не требуется Q/K/V и softmax Глобальный охват токенов.

* Матрица token-mixing всё ещё $O(L^2)$ и привязана к фиксированному числу патчей $L$ — не переносится на другое разрешение без переобучения.
* Требуется много данных.

ConvMixer:

* Линейная сложность $O(L\cdot k^2 d)$, не зависит от $L$.

* Предполагает, что пиксели, находящиеся рядом, связаны и поэтому лучше обучается на датасетах меньшего размера.

* Естественно работает с любым разрешением входа.

* Ограниченное рецептивное поле; дальние зависимости требуют большего $k$ или большей глубины.
* Нет динамической адаптации к содержимому входа.